# MVP4: Multi-Dataset Benchmark

**3 model families × 6 datasets**

| Family | Models | Train sizes |
|--------|--------|-------------|
| ML | XGBoost+TF-IDF, SVM+TF-IDF | XGB: 100/1000/5000, SVM: 100/1000 |
| ~BERT | SetFit (all-MiniLM-L6-v2) | 100/1000 |
| LLM | gemini-3.1-flash-lite-preview, gemma-3-4b-it | 30 train / 30 test |

**Datasets:** ag_news, reward-bench, glue (sst2), tweet_eval (sentiment), social-bias-frames, lex_glue (scotus)

Test size: `min(available, 5000)` for ML/BERT, `30` for LLMs. Stratified when possible.

In [ ]:
! pip install -q datasets scikit-learn xgboost "setfit>=1.0" "transformers<5.0" openai pandas numpy tqdm

In [ ]:
from __future__ import annotations

import os, sys, json, gc, traceback, warnings
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# SEED = 42
np.random.seed(SEED)

# Kaggle-compatible output directory
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Global results accumulator
ALL_RESULTS: list[dict] = []

def now_stamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def save_result(result: dict):
    """Append a result dict and save incrementally to JSON."""
    ALL_RESULTS.append(result)
    path = RESULTS_DIR / "results_incremental.json"
    path.write_text(json.dumps(ALL_RESULTS, indent=2, default=str))
    status = result.get("error", None)
    tag = "ERR" if status else "OK"
    print(f"  [{tag}] Saved {len(ALL_RESULTS)} results -> {path.name}")

## API Key Setup
Set your Google AI API key below for LLM inference.

In [ ]:
# ==================== API KEY SETUP ====================
# Replace with your actual Google AI API key
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# =======================================================

## Dataset Registry & Loading

In [ ]:
from datasets import load_dataset, Features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATASET_CONFIGS = [
    {
        "name": "ag_news",
        "hf_path": "ag_news",
        "hf_config": None,
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "reward_bench",
        "hf_path": "allenai/reward-bench",
        "hf_config": None,
        "train_split": "filtered",
        "test_split": None,  # single split -> manual split
        "text_col": "prompt",
        "label_col": "subset",
    },
    {
        "name": "glue_sst2",
        "hf_path": "glue",
        "hf_config": "sst2",
        "train_split": "train",
        "test_split": "validation",  # test labels are -1
        "text_col": "sentence",
        "label_col": "label",
    },
    {
        "name": "tweet_eval_sentiment",
        "hf_path": "tweet_eval",
        "hf_config": "sentiment",
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "social_bias_frames",
        "hf_path": "social_bias_frames",
        "hf_config": None,
        "train_split": "train",
        "test_split": "test",
        "text_col": "post",
        "label_col": "offensiveYN",
        "binarize": True,
    },
    {
        "name": "lex_glue_scotus",
        "hf_path": "lex_glue",
        "hf_config": "scotus",
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
]


def _resolve_label_names(ds_or_df, label_col, hf_dataset=None):
    """Try to get human-readable label names from HF features, fall back to unique values."""
    if hf_dataset is not None:
        try:
            feat = hf_dataset.features[label_col]
            if hasattr(feat, "names"):
                return feat.names
        except Exception:
            pass
    if isinstance(ds_or_df, pd.DataFrame):
        return sorted(ds_or_df[label_col].dropna().unique().astype(str).tolist())
    return None


def load_and_prepare_dataset(config: dict):
    """Load an HF dataset, normalise to (text, label) DataFrame pair.

    Returns: (train_df, test_df, label_names)
    """
    name = config["name"]
    print(f"\n{'='*60}\nLoading dataset: {name}\n{'='*60}")

    kw = {"path": config["hf_path"]}
    if config.get("hf_config"):
        kw["name"] = config["hf_config"]

    text_col = config["text_col"]
    label_col = config["label_col"]

    # --- Load ---
    if config["test_split"] is None:
        raw = load_dataset(**kw, split=config["train_split"])
        label_names_raw = _resolve_label_names(None, label_col, raw)
        df = raw.to_pandas()[[text_col, label_col]].dropna().reset_index(drop=True)
        df[text_col] = df[text_col].astype(str).str.strip()
        df[label_col] = df[label_col].astype(str)
        try:
            train_df, test_df = train_test_split(
                df, test_size=0.3, random_state=SEED, stratify=df[label_col]
            )
        except ValueError:
            train_df, test_df = train_test_split(df, test_size=0.3, random_state=SEED)
        train_df = train_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
    else:
        train_raw = load_dataset(**kw, split=config["train_split"])
        test_raw = load_dataset(**kw, split=config["test_split"])
        label_names_raw = _resolve_label_names(None, label_col, train_raw)
        train_df = train_raw.to_pandas()[[text_col, label_col]].dropna().reset_index(drop=True)
        test_df = test_raw.to_pandas()[[text_col, label_col]].dropna().reset_index(drop=True)
        train_df[text_col] = train_df[text_col].astype(str).str.strip()
        test_df[text_col] = test_df[text_col].astype(str).str.strip()

    # --- Binarize if needed (social_bias_frames) ---
    if config.get("binarize"):
        for _df in [train_df, test_df]:
            _df[label_col] = _df[label_col].apply(
                lambda x: "offensive" if float(x) >= 0.5 else "not_offensive"
            )
    else:
        # Map integer labels to names when available
        if label_names_raw and train_df[label_col].dtype != object:
            mapping = {str(i): n for i, n in enumerate(label_names_raw)}
            train_df[label_col] = train_df[label_col].astype(str).map(mapping).fillna(train_df[label_col].astype(str))
            test_df[label_col] = test_df[label_col].astype(str).map(mapping).fillna(test_df[label_col].astype(str))
        else:
            train_df[label_col] = train_df[label_col].astype(str)
            test_df[label_col] = test_df[label_col].astype(str)

    # --- Rename to standard schema ---
    train_df = train_df.rename(columns={text_col: "text", label_col: "label"})
    test_df = test_df.rename(columns={text_col: "text", label_col: "label"})

    # Remove empty texts
    train_df = train_df[train_df["text"].str.len() > 0].reset_index(drop=True)
    test_df = test_df[test_df["text"].str.len() > 0].reset_index(drop=True)

    label_names = sorted(set(train_df["label"].tolist()) | set(test_df["label"].tolist()))

    print(f"  Train: {len(train_df):,}, Test: {len(test_df):,}, Labels ({len(label_names)}): {label_names[:10]}")
    print(f"  Test dist: {test_df['label'].value_counts().head(5).to_dict()}")
    return train_df, test_df, label_names

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report


def sample_data(train_df, test_df, *, train_n, test_n, stratify=True):
    """Draw stratified samples from train/test DataFrames."""
    # --- Test sample ---
    if len(test_df) > test_n:
        try:
            if stratify:
                ts, _ = train_test_split(test_df, train_size=test_n,
                                         random_state=SEED, stratify=test_df["label"])
            else:
                ts = test_df.sample(n=test_n, random_state=SEED)
        except ValueError:
            ts = test_df.sample(n=test_n, random_state=SEED)
    else:
        ts = test_df.copy()

    # --- Train sample ---
    if len(train_df) > train_n:
        try:
            if stratify:
                tr, _ = train_test_split(train_df, train_size=train_n,
                                         random_state=SEED, stratify=train_df["label"])
            else:
                tr = train_df.sample(n=train_n, random_state=SEED)
        except ValueError:
            tr = train_df.sample(n=train_n, random_state=SEED)
    else:
        tr = train_df.copy()

    return tr.reset_index(drop=True), ts.reset_index(drop=True)


def evaluate(y_true, y_pred, label_names):
    """Compute accuracy, macro-F1, weighted-F1 and a full report."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "report": classification_report(y_true, y_pred, labels=label_names,
                                        zero_division=0, output_dict=True),
    }

## 1. ML Models — XGBoost + TF-IDF / SVM + TF-IDF

- XGBoost train sizes: 100, 1 000, 5 000
- SVM train sizes: 100, 1 000
- Test: min(available, 5 000)  •  Stratified

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

ML_TEST_N = 5000


def run_ml_experiment(train_df, test_df, label_names, dataset_name,
                      model_type, train_n):
    le = LabelEncoder()
    le.fit(label_names)

    tr, ts = sample_data(train_df, test_df, train_n=train_n, test_n=ML_TEST_N)

    X_train = tr["text"].tolist()
    y_train = le.transform(tr["label"].tolist())
    X_test = ts["text"].tolist()
    y_true = ts["label"].tolist()

    tfidf = TfidfVectorizer(
        lowercase=True, strip_accents="unicode",
        ngram_range=(1, 2), min_df=2, max_features=80_000,
    )

    if model_type == "xgboost":
        n_cls = len(label_names)
        clf = XGBClassifier(
            objective="multi:softprob" if n_cls > 2 else "binary:logistic",
            eval_metric="mlogloss", n_estimators=350, max_depth=6,
            learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, tree_method="hist",
            device="cuda" if device == "cuda" else "cpu",
            random_state=SEED, n_jobs=-1,
        )
    else:  # svm
        clf = LinearSVC(random_state=SEED, max_iter=10_000)

    pipe = Pipeline([("tfidf", tfidf), ("clf", clf)])

    t0 = perf_counter()
    pipe.fit(X_train, y_train)
    fit_s = perf_counter() - t0

    t0 = perf_counter()
    y_pred_enc = pipe.predict(X_test)
    inf_s = perf_counter() - t0

    y_pred = le.inverse_transform(y_pred_enc).tolist()
    metrics = evaluate(y_true, y_pred, label_names)

    result = {
        "dataset": dataset_name, "model_family": "ML",
        "model": f"{model_type}_tfidf",
        "train_n": int(len(tr)), "test_n": int(len(ts)),
        "fit_seconds": round(fit_s, 2), "infer_seconds": round(inf_s, 2),
        **{k: round(v, 4) for k, v in metrics.items() if k != "report"},
    }

    pred_path = RESULTS_DIR / f"preds_{dataset_name}_{model_type}_n{train_n}.csv"
    pd.DataFrame({"text": X_test, "true": y_true, "pred": y_pred}).to_csv(pred_path, index=False)

    print(f"  {model_type} n_train={len(tr)} n_test={len(ts)} "
          f"acc={metrics['accuracy']:.4f} F1={metrics['macro_f1']:.4f} fit={fit_s:.1f}s")
    return result


# ---------- Run all ML experiments ----------
for ds_cfg in DATASET_CONFIGS:
    try:
        train_df, test_df, label_names = load_and_prepare_dataset(ds_cfg)

        for n in [100, 1000, 5000]:
            try:
                save_result(run_ml_experiment(
                    train_df, test_df, label_names, ds_cfg["name"], "xgboost", n))
            except Exception as exc:
                print(f"  ERROR xgboost n={n}: {exc}")
                traceback.print_exc()
                save_result({"dataset": ds_cfg["name"], "model": "xgboost_tfidf",
                             "train_n": n, "error": str(exc)})

        for n in [100, 1000]:
            try:
                save_result(run_ml_experiment(
                    train_df, test_df, label_names, ds_cfg["name"], "svm", n))
            except Exception as exc:
                print(f"  ERROR svm n={n}: {exc}")
                traceback.print_exc()
                save_result({"dataset": ds_cfg["name"], "model": "svm_tfidf",
                             "train_n": n, "error": str(exc)})

        del train_df, test_df
        gc.collect()
    except Exception as exc:
        print(f"ERROR loading {ds_cfg['name']}: {exc}")
        traceback.print_exc()

## 2. ~BERT Models — SetFit (sentence-transformers/all-MiniLM-L6-v2)

- Train sizes: 100, 1 000
- Test: min(available, 5 000)
- `max_steps = 10 000`

In [ ]:
from datasets import Dataset as HFDataset
from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitArgs

SETFIT_TEST_N = 5000
SETFIT_MAX_STEPS = 10_000


def run_setfit_experiment(train_df, test_df, label_names, dataset_name, train_n):
    le = LabelEncoder()
    le.fit(label_names)

    tr, ts = sample_data(train_df, test_df, train_n=train_n, test_n=SETFIT_TEST_N)

    train_ds = HFDataset.from_pandas(
        pd.DataFrame({"text": tr["text"], "label": le.transform(tr["label"])}),
        preserve_index=False,
    )

    label_ids = list(range(len(label_names)))
    model = SetFitModel.from_pretrained(
        "sentence-transformers/all-MiniLM-L6-v2", labels=label_ids
    ).to(device)

    args = SetFitArgs(
        batch_size=16, num_epochs=1,
        max_steps=SETFIT_MAX_STEPS,
        evaluation_strategy="no", save_strategy="no",
    )

    trainer = SetFitTrainer(model=model, args=args, train_dataset=train_ds)

    print(f"  SetFit training n={len(tr)} max_steps={SETFIT_MAX_STEPS} ...")
    t0 = perf_counter()
    trainer.train()
    fit_s = perf_counter() - t0

    test_texts = ts["text"].tolist()
    y_true = ts["label"].tolist()

    t0 = perf_counter()
    pred_ids = [int(p) for p in model.predict(test_texts, use_labels=False)]
    inf_s = perf_counter() - t0
    y_pred = le.inverse_transform(pred_ids).tolist()

    metrics = evaluate(y_true, y_pred, label_names)

    result = {
        "dataset": dataset_name, "model_family": "BERT",
        "model": "setfit_MiniLM",
        "train_n": int(len(tr)), "test_n": int(len(ts)),
        "fit_seconds": round(fit_s, 2), "infer_seconds": round(inf_s, 2),
        **{k: round(v, 4) for k, v in metrics.items() if k != "report"},
    }

    pred_path = RESULTS_DIR / f"preds_{dataset_name}_setfit_n{train_n}.csv"
    pd.DataFrame({"text": test_texts, "true": y_true, "pred": y_pred}).to_csv(
        pred_path, index=False
    )

    print(f"  setfit n_train={len(tr)} n_test={len(ts)} "
          f"acc={metrics['accuracy']:.4f} F1={metrics['macro_f1']:.4f} fit={fit_s:.1f}s")

    del model, trainer, train_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ---------- Run all SetFit experiments ----------
for ds_cfg in DATASET_CONFIGS:
    try:
        train_df, test_df, label_names = load_and_prepare_dataset(ds_cfg)
        for n in [100, 1000]:
            try:
                save_result(run_setfit_experiment(
                    train_df, test_df, label_names, ds_cfg["name"], n))
            except Exception as exc:
                print(f"  ERROR setfit n={n}: {exc}")
                traceback.print_exc()
                save_result({"dataset": ds_cfg["name"], "model": "setfit_MiniLM",
                             "train_n": n, "error": str(exc)})
        del train_df, test_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as exc:
        print(f"ERROR loading {ds_cfg['name']}: {exc}")
        traceback.print_exc()

## 3. LLM Models

- Models: `gemini-3.1-flash-lite-preview`, `gemma-3-4b-it`
- 30 train (few-shot) / 30 test per dataset
- OpenAI client → Google AI endpoint
- No system prompt, manual JSON handling
- `max_retries = 30`, batch prediction

In [ ]:
from openai import OpenAI

LLM_MODELS = [
    "gemini-3.1-flash-lite-preview",
    "gemma-3-4b-it",
]
LLM_TRAIN_N = 30
LLM_TEST_N = 30
LLM_FEW_SHOT_IN_PROMPT = 5  # how many examples to include in prompt
MAX_TEXT_CHARS = 800  # truncate long texts for prompts


def build_classification_prompt(text, label_names, few_shot_examples=None):
    """Build a zero/few-shot classification prompt.
    No system prompt. Manual JSON handling (plain text answer expected).
    """
    parts = [
        f"Classify the following text into exactly one of these categories: "
        f"{', '.join(label_names)}.",
        "",
        "Return ONLY the category name, nothing else.",
        "",
    ]

    if few_shot_examples:
        parts.append("Here are some examples:")
        parts.append("")
        for ex in few_shot_examples:
            t = ex["text"][:MAX_TEXT_CHARS]
            parts.append(f'Text: "{t}"')
            parts.append(f"Category: {ex['label']}")
            parts.append("")

    trunc = text[:MAX_TEXT_CHARS]
    parts.append(f'Now classify this text:')
    parts.append(f'Text: "{trunc}"')
    parts.append(f"Category:")

    return "\n".join(parts)


def parse_llm_label(response_text, label_names):
    """Extract a valid label from LLM response. Handles plain text & JSON."""
    raw = response_text.strip()

    # Direct match (exact or case-insensitive)
    for label in label_names:
        if raw.lower() == label.lower():
            return label

    # Substring match (first found)
    raw_lower = raw.lower()
    for label in label_names:
        if label.lower() in raw_lower:
            return label

    # Try JSON parse
    try:
        data = json.loads(raw)
        if isinstance(data, dict):
            for key in ("category", "label", "class", "answer", "prediction"):
                if key in data:
                    val = str(data[key]).strip()
                    for label in label_names:
                        if label.lower() == val.lower():
                            return label
    except (json.JSONDecodeError, TypeError):
        pass

    # Fallback: return the closest label (first in list)
    return label_names[0]


def run_llm_experiment(train_df, test_df, label_names, dataset_name, model_name):
    """Run LLM classification: few-shot prompting via OpenAI-compatible API."""
    tr, ts = sample_data(train_df, test_df,
                         train_n=LLM_TRAIN_N, test_n=LLM_TEST_N, stratify=True)

    few_shot = tr[["text", "label"]].head(LLM_FEW_SHOT_IN_PROMPT).to_dict("records")

    # OpenAI client pointing at Google endpoint, max_retries=30
    client = OpenAI(
        api_key=GOOGLE_API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        max_retries=30,
    )

    y_true = ts["label"].tolist()
    y_pred = []
    errors = 0

    t0 = perf_counter()

    # Batch prediction: iterate all test samples
    for idx, row in tqdm(ts.iterrows(), total=len(ts), desc=f"{model_name[:20]}"):
        prompt = build_classification_prompt(row["text"], label_names,
                                             few_shot_examples=few_shot)
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=64,
            )
            answer = resp.choices[0].message.content or ""
            pred = parse_llm_label(answer, label_names)
        except Exception as exc:
            pred = label_names[0]
            errors += 1
            if errors <= 3:
                print(f"    LLM call error #{errors}: {exc}")

        y_pred.append(pred)

    inf_s = perf_counter() - t0
    metrics = evaluate(y_true, y_pred, label_names)

    result = {
        "dataset": dataset_name, "model_family": "LLM",
        "model": model_name,
        "train_n": int(len(tr)), "test_n": int(len(ts)),
        "few_shot_in_prompt": len(few_shot),
        "infer_seconds": round(inf_s, 2), "api_errors": errors,
        **{k: round(v, 4) for k, v in metrics.items() if k != "report"},
    }

    pred_path = RESULTS_DIR / f"preds_{dataset_name}_{model_name.replace('/', '_')}.csv"
    pd.DataFrame({"text": ts["text"].tolist(), "true": y_true, "pred": y_pred}).to_csv(
        pred_path, index=False
    )

    print(f"  {model_name} n_train={len(tr)} n_test={len(ts)} "
          f"acc={metrics['accuracy']:.4f} F1={metrics['macro_f1']:.4f} "
          f"errors={errors} time={inf_s:.1f}s")
    return result


# ---------- Run all LLM experiments ----------
for ds_cfg in DATASET_CONFIGS:
    try:
        train_df, test_df, label_names = load_and_prepare_dataset(ds_cfg)

        for model_name in LLM_MODELS:
            try:
                save_result(run_llm_experiment(
                    train_df, test_df, label_names, ds_cfg["name"], model_name))
            except Exception as exc:
                print(f"  ERROR LLM {model_name}: {exc}")
                traceback.print_exc()
                save_result({"dataset": ds_cfg["name"], "model": model_name,
                             "model_family": "LLM", "error": str(exc)})

        del train_df, test_df
        gc.collect()
    except Exception as exc:
        print(f"ERROR loading {ds_cfg['name']}: {exc}")
        traceback.print_exc()

## Final Results Summary

In [ ]:
# ---- Save final consolidated results ----
results_df = pd.DataFrame(ALL_RESULTS)
csv_path = RESULTS_DIR / "mvp4_final_results.csv"
json_path = RESULTS_DIR / "mvp4_final_results.json"

results_df.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(ALL_RESULTS, indent=2, default=str))

print(f"Total experiments: {len(ALL_RESULTS)}")
print(f"CSV:  {csv_path}")
print(f"JSON: {json_path}")

# ---- Display summary table ----
show_cols = ["dataset", "model_family", "model", "train_n", "test_n",
             "accuracy", "macro_f1", "weighted_f1"]
avail = [c for c in show_cols if c in results_df.columns]
if len(results_df) > 0:
    print("\n" + results_df[avail].to_string(index=False))
else:
    print("No results collected.")

In [ ]:
# ---- Pivot: macro_f1 by dataset × model ----
if len(results_df) > 0 and "macro_f1" in results_df.columns:
    results_df["model_tag"] = (
        results_df["model"] + "_n" + results_df.get("train_n", 0).astype(str)
    )
    pivot = results_df.pivot_table(
        index="dataset", columns="model_tag", values="macro_f1", aggfunc="first"
    )
    print("\nMacro-F1 Pivot (dataset × model_tag):\n")
    print(pivot.round(4).to_string())

    pivot_path = RESULTS_DIR / "mvp4_pivot_f1.csv"
    pivot.to_csv(pivot_path)
    print(f"\nSaved pivot to {pivot_path}")